# Fake News Detection — Nível 2: Feature Engineering Estilométrico + XGBoost

---

## Motivação

No **Nível 1**, mostramos que representações TF-IDF com modelos lineares atingem F1 Macro ≈ 0.9939. Esses modelos aprendem *o quê* está escrito (vocabulário), mas ignoram *como* está escrito (estilo).

**Hipótese do Nível 2:** notícias falsas e verdadeiras apresentam padrões estilísticos distintos — uso de pontuação, riqueza de vocabulário, estrutura de sentenças, capitalização — que são capturáveis como features numéricas independentes do vocabulário.

---

## O que é Estilometria?

**Estilometria** é o estudo quantitativo do estilo de escrita. Originalmente usada em análise de autoria literária (quem escreveu Shakespeare?), aplica-se aqui para distinguir o estilo jornalístico formal (Reuters) do estilo informal/sensacionalista de sites de fake news.

Features estilométricas são:
- **Independentes do vocabulário**: um artigo pode usar palavras novas mas manter o mesmo estilo
- **Culturalmente estáveis**: enquanto o vocabulário muda com o tempo, hábitos de pontuação e capitalização mudam muito menos
- **Interpretáveis**: cada feature tem significado direto e verificável

---

## Por que XGBoost?

Features estilométricas são **heterogêneas**: contagens, ratios, booleans — com escalas e distribuições muito diferentes. Modelos lineares tratam todas as features simetricamente, o que é subótimo aqui. O **XGBoost (Extreme Gradient Boosting)** é ideal porque:

1. **Invariância de escala**: árvores de decisão usam thresholds ordinais — não precisam de normalização
2. **Interações automáticas**: captura relações não-lineares entre features (ex: `caps_ratio` alto E `exclamation_count` alto → quase certamente fake)
3. **Robustez a outliers**: splits binários são menos afetados por valores extremos
4. **SHAP nativo**: o TreeExplainer do SHAP é exato para modelos de árvore

---

## Estrutura do Notebook

| # | Seção | Conteúdo |
|---|-------|----------|
| 1 | Setup | Imports e configurações globais |
| 2 | Dados | Carregamento, pré-processamento (=Nível 1) |
| 3 | Estilometria | Teoria e extração de 22 features |
| 4 | EDA Estilométrica | Visualizações por classe |
| 5 | Split + TF-IDF | Particionamento e vetorização |
| 6 | XGBoost Estilométrico | Modelo A: apenas features de estilo |
| 7 | XGBoost Híbrido | Modelo B: TF-IDF + estilométrico |
| 8 | SHAP | Explicabilidade do Modelo A |
| 9 | Comparação | Benchmark contra Nível 1 |
| 10 | Conclusões | Resumo, limitações, próximos passos |

## 1. Configuração do Ambiente

In [ ]:
# ── Bibliotecas padrão ──────────────────────────────────────────────────────
import os
import re
import time
import warnings
warnings.filterwarnings('ignore')

# ── Dados ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.sparse as sp

# ── Visualização ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── NLP ──────────────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    f1_score, accuracy_score, precision_score, recall_score
)

# ── XGBoost e SHAP ────────────────────────────────────────────────────────────
import xgboost as xgb
import shap

# ── Caminhos ─────────────────────────────────────────────────────────────────
from pathlib import Path

# ── Configurações globais ─────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PALETTE = {
    'fake':    '#E74C3C',   # vermelho — classe 0
    'real':    '#2ECC71',   # verde    — classe 1
    'neutral': '#3498DB',   # azul     — uso geral
}

RESULTS_DIR = Path('../results/level2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_style('whitegrid')

# ── Downloads NLTK ────────────────────────────────────────────────────────────
nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))

print('Ambiente configurado.')
print(f'  NumPy        : {np.__version__}')
print(f'  Pandas       : {pd.__version__}')
print(f'  Scikit-learn : {__import__("sklearn").__version__}')
print(f'  XGBoost      : {xgb.__version__}')
print(f'  SHAP         : {shap.__version__}')
print(f'Figuras serão salvas em: {RESULTS_DIR.resolve()}')

## 2. Carregamento e Pré-processamento

Reutilizamos exatamente o mesmo pipeline do Nível 1:
- Concatenação de `Fake.csv` (label=0) e `True.csv` (label=1)
- Remoção da byline Reuters (leakage crítico)
- Features: `title` + `text` (descartamos `subject` e `date`)

**Diferença do Nível 1:** mantemos duas versões do texto:
1. `full_text` — preprocessado (sem pontuação, lowercase) → para TF-IDF
2. `title` e `text` brutos (apenas byline removida) → para extração estilométrica

In [ ]:
DATA_DIR = '../data'

df_fake = pd.read_csv(os.path.join(DATA_DIR, 'Fake.csv'))
df_real = pd.read_csv(os.path.join(DATA_DIR, 'True.csv'))

df_fake['label'] = 0
df_real['label'] = 1

df = (pd.concat([df_fake, df_real], ignore_index=True)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True))

print(f'Total de artigos : {len(df):,}')
print(f'Fake (0)         : {(df["label"]==0).sum():,}')
print(f'Real (1)         : {(df["label"]==1).sum():,}')

In [ ]:
# Padrão da byline Reuters (igual ao Nível 1)
REUTERS_RE = re.compile(
    r'^[A-Z][A-Z\s,\.]{2,40}\(Reuters\)\s*[-\u2013]\s*',
    re.IGNORECASE
)

def preprocess_tfidf(text):
    """Pipeline de limpeza para TF-IDF (idêntico ao Nível 1)."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = REUTERS_RE.sub('', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)


print('Aplicando pré-processamento para TF-IDF...')
t0 = time.time()

df['title_clean'] = df['title'].apply(preprocess_tfidf)
df['text_clean']  = df['text'].apply(preprocess_tfidf)
df['full_text']   = df['title_clean'] + ' titleend ' + df['text_clean']

# Remove byline Reuters do texto bruto (para estilometria sem leakage)
df['text_nr'] = df['text'].astype(str).apply(
    lambda t: REUTERS_RE.sub('', t)
)

print(f'Concluído em {time.time()-t0:.1f}s')

## 3. Feature Engineering Estilométrico

### Teoria: O Que Capturar?

A hipótese central é que **fake news e notícias reais têm estilos de escrita sistematicamente diferentes**:

| Dimensão | Real (Reuters) | Fake (sites alternativos) |
|----------|---------------|---------------------------|
| **Vocabulário** | Diverso, técnico, formal | Repetitivo, emotivo, coloquial |
| **Sentenças** | Longas, complexas | Curtas, fragmentadas ou excessivamente longas |
| **Pontuação** | Convencional | Exclamações, reticências, abuso de maiúsculas |
| **Capitalização** | Apenas nomes próprios | ALL CAPS para ênfase emocional |
| **Números** | Dados, estatísticas | Raros ou usados retoricamente |

---

### Features Implementadas

#### Grupo A — Riqueza Lexical

**Type-Token Ratio (TTR):** proporção de palavras únicas sobre o total:

$$\text{TTR} = \frac{|\{\text{tipos}\}|}{|\{\text{tokens}\}|}$$

Um TTR maior indica vocabulário mais rico e variado — característica do jornalismo formal.

**Comprimento médio de palavra:**

$$\bar{L}_{\text{word}} = \frac{1}{N} \sum_{i=1}^{N} |w_i|$$

Palavras mais longas indicam linguagem técnica/formal (ex: *"investigation"* vs *"probe"*).

#### Grupo B — Estrutura de Sentenças

Segmentamos as sentenças por `.`, `!`, `?`. Para cada artigo:

- **`sent_count`**: quantidade de sentenças
- **`avg_sent_len`**: comprimento médio ($\mu$)
- **`std_sent_len`**: desvio padrão do comprimento ($\sigma$) — alta variância pode indicar texto fragmentado

#### Grupo C — Pontuação e Emoção

- **`exclamation_count`**: exclamações — marcador de sensacionalismo
- **`question_count`**: perguntas retóricas — técnica comum em fake news
- **`ellipsis_count`**: reticências — suspense artificial
- **`comma_ratio`**: vírgulas por palavra — proxy de complexidade sintática

#### Grupo D — Capitalização

- **`caps_ratio`**: proporção de letras maiúsculas sobre total de letras
- **`caps_word_ratio`**: proporção de palavras completamente em MAIÚSCULAS

#### Grupo E — Título

O título é processado separadamente pois é o primeiro ponto de contato do leitor — e o principal veículo de clickbait:
- Comprimento, riqueza lexical, capitalização, presença de `!` e `?`

In [ ]:
def extract_stylometric(title, text_nr):
    """Extrai 22 features estilométricas de um artigo."""
    title   = str(title)   if isinstance(title,   str) else ''
    text_nr = str(text_nr) if isinstance(text_nr, str) else ''

    feats = {}

    # ── GRUPO A: Riqueza Lexical (corpo do artigo) ────────────────────────────
    words        = text_nr.split()
    words_lower  = [w.lower() for w in words]
    alpha_words  = [re.sub(r'[^a-zA-Z]', '', w) for w in words]
    alpha_words  = [w for w in alpha_words if w]

    n_words = max(len(words), 1)

    feats['word_count']        = len(words)
    feats['unique_word_ratio'] = len(set(words_lower)) / n_words  # TTR
    feats['avg_word_len']      = (
        np.mean([len(w) for w in alpha_words]) if alpha_words else 0.0
    )

    # ── GRUPO B: Estrutura de Sentenças ───────────────────────────────────────
    sents     = [s.strip() for s in re.split(r'[.!?]+', text_nr) if s.strip()]
    sent_lens = [len(s.split()) for s in sents]

    feats['sent_count']   = len(sents)
    feats['avg_sent_len'] = np.mean(sent_lens) if sent_lens else 0.0
    feats['std_sent_len'] = np.std(sent_lens)  if len(sent_lens) > 1 else 0.0

    # ── GRUPO C: Pontuação e Emoção ───────────────────────────────────────────
    # Usamos o texto original (com pontuação) para contar estes marcadores
    raw = str(text_nr)
    feats['exclamation_count'] = raw.count('!')
    feats['question_count']    = raw.count('?')
    feats['ellipsis_count']    = raw.count('...')
    feats['comma_ratio']       = raw.count(',') / n_words
    feats['quote_count']       = raw.count('"') + raw.count('\u201c') + raw.count('\u201d')
    feats['number_ratio']      = len(re.findall(r'\b\d+\b', raw)) / n_words
    feats['url_count']         = len(re.findall(r'https?://\S+|www\.\S+', raw))

    # ── GRUPO D: Capitalização ────────────────────────────────────────────────
    alpha_chars = [c for c in raw if c.isalpha()]
    n_alpha     = max(len(alpha_chars), 1)
    feats['caps_ratio']      = sum(1 for c in alpha_chars if c.isupper()) / n_alpha
    feats['caps_word_ratio'] = (
        sum(1 for w in words if w.isupper() and len(w) > 1) / n_words
    )

    # ── GRUPO E: Features do Título ───────────────────────────────────────────
    title_words  = title.split()
    title_alpha  = [re.sub(r'[^a-zA-Z]', '', w) for w in title_words]
    title_alpha  = [w for w in title_alpha if w]
    n_title      = max(len(title_words), 1)

    feats['title_word_count']      = len(title_words)
    feats['title_char_count']      = len(title)
    feats['title_avg_word_len']    = (
        np.mean([len(w) for w in title_alpha]) if title_alpha else 0.0
    )
    feats['title_caps_ratio']      = (
        sum(1 for c in title if c.isupper()) /
        max(sum(1 for c in title if c.isalpha()), 1)
    )
    feats['title_caps_word_ratio'] = (
        sum(1 for w in title_words if w.isupper() and len(w) > 1) / n_title
    )
    feats['title_has_exclamation'] = int('!' in title)
    feats['title_has_question']    = int('?' in title)

    # ── COMBINADA: relação título/texto ───────────────────────────────────────
    feats['title_word_ratio'] = len(title_words) / n_words

    return feats


print('Extraindo features estilométricas...')
t0 = time.time()

styl_records = [
    extract_stylometric(row['title'], row['text_nr'])
    for _, row in df.iterrows()
]

df_styl = pd.DataFrame(styl_records)
df_styl['label'] = df['label'].values

STYL_FEATURES = [c for c in df_styl.columns if c != 'label']

print(f'Concluído em {time.time()-t0:.1f}s')
print(f'Features extraídas : {len(STYL_FEATURES)}')
print(f'Nomes: {STYL_FEATURES}')

## 4. Análise Exploratória das Features Estilométricas

Antes de modelar, visualizamos a **separabilidade** de cada feature entre as classes Fake e Real. Uma feature com distribuições bem separadas entre as classes tem alto poder discriminativo e será valorizada pelo modelo.

In [ ]:
# ── Estatísticas descritivas por classe ───────────────────────────────────────
print('=== Médias por Classe (Fake=0, Real=1) ===')
summary = (df_styl.groupby('label')[STYL_FEATURES]
           .mean()
           .T
           .rename(columns={0: 'Fake', 1: 'Real'}))
summary['Delta (Real-Fake)'] = summary['Real'] - summary['Fake']
summary['Delta%'] = (summary['Delta (Real-Fake)'] / summary['Fake'].abs().replace(0, 1) * 100).round(1)
print(summary.to_string(float_format='{:.4f}'.format))

In [ ]:
# ── Box plots das principais features por classe ──────────────────────────────
# Selecionamos 12 features visualmente mais informativas
PLOT_FEATURES = [
    'unique_word_ratio', 'avg_word_len', 'avg_sent_len', 'std_sent_len',
    'exclamation_count', 'caps_ratio', 'caps_word_ratio', 'url_count',
    'title_caps_word_ratio', 'title_has_exclamation', 'number_ratio', 'comma_ratio'
]

FEATURE_LABELS = {
    'unique_word_ratio':     'TTR (vocab. richness)',
    'avg_word_len':          'Avg word length',
    'avg_sent_len':          'Avg sentence length',
    'std_sent_len':          'Sentence len std',
    'exclamation_count':     'Exclamation count',
    'caps_ratio':            'CAPS char ratio',
    'caps_word_ratio':       'ALL-CAPS word ratio',
    'url_count':             'URL count',
    'title_caps_word_ratio': 'Title ALL-CAPS ratio',
    'title_has_exclamation': 'Title has "!"',
    'number_ratio':          'Number ratio',
    'comma_ratio':           'Comma ratio',
}

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

df_plot = df_styl.copy()
df_plot['Classe'] = df_plot['label'].map({0: 'Fake', 1: 'Real'})
class_colors = ['Fake', 'Real']

for i, feat in enumerate(PLOT_FEATURES):
    ax = axes[i]

    # Clip outliers para legibilidade (percentil 99)
    clip_val = df_plot[feat].quantile(0.99)
    plot_data = df_plot.copy()
    plot_data[feat] = plot_data[feat].clip(upper=clip_val)

    sns.boxplot(
        data=plot_data,
        x='Classe', y=feat,
        palette={'Fake': PALETTE['fake'], 'Real': PALETTE['real']},
        order=['Fake', 'Real'],
        width=0.5, linewidth=0.8,
        ax=ax, fliersize=1
    )
    ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distribuição das Features Estilométricas por Classe (outliers clipados no P99)',
             fontsize=13, y=1.01)
plt.tight_layout()

plt.savefig(RESULTS_DIR / 'eda_stylometric.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Particionamento e Vetorização

Mantemos **exatamente o mesmo split** do Nível 1 (`random_state=42`, estratificado, 70/15/15%) para que a comparação de métricas seja válida — os conjuntos de teste são idênticos.

In [ ]:
# ── Arrays de features e target ───────────────────────────────────────────────
X_text = df['full_text'].values   # para TF-IDF
X_styl = df_styl[STYL_FEATURES].values.astype(np.float32)  # estilométrico
y      = df['label'].values

# ── Split 70 / 15 / 15 ───────────────────────────────────────────────────────
indices = np.arange(len(y))

idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

# ── TF-IDF (idêntico ao Nível 1) ─────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), max_features=100_000,
    sublinear_tf=True, min_df=3, max_df=0.95,
    norm='l2', analyzer='word', dtype=np.float32
)

X_train_tfidf = tfidf.fit_transform(X_text[idx_train])
X_val_tfidf   = tfidf.transform(X_text[idx_val])
X_test_tfidf  = tfidf.transform(X_text[idx_test])

# ── Matrizes estilométricas ───────────────────────────────────────────────────
X_train_styl = X_styl[idx_train]
X_val_styl   = X_styl[idx_val]
X_test_styl  = X_styl[idx_test]

# ── Matrizes híbridas: TF-IDF + estilométrico ─────────────────────────────────
X_train_hybrid = sp.hstack([X_train_tfidf, sp.csr_matrix(X_train_styl)], format='csr')
X_val_hybrid   = sp.hstack([X_val_tfidf,   sp.csr_matrix(X_val_styl)],   format='csr')
X_test_hybrid  = sp.hstack([X_test_tfidf,  sp.csr_matrix(X_test_styl)],  format='csr')

print('=== Shapes das Matrizes ===')
for name, m in [('TF-IDF treino', X_train_tfidf), ('Estilom. treino', X_train_styl),
                ('Híbrida treino', X_train_hybrid), ('Teste', X_test_tfidf)]:
    print(f'  {name:<20}: {m.shape}')

## 6. XGBoost — Teoria

### Gradient Boosting

O **Gradient Boosting** constrói um ensemble de árvores de decisão **sequencialmente**, onde cada nova árvore corrige os erros da anterior. Diferente do Bagging (Random Forest), que treina árvores em paralelo sobre amostras aleatórias, o Boosting é um processo adaptativo.

O modelo final é uma soma ponderada de $T$ árvores:

$$\hat{y}_i = \sum_{t=1}^{T} f_t(\mathbf{x}_i)$$

onde cada $f_t$ é uma árvore de decisão rasa (tipicamente `max_depth` de 3 a 8).

### O Passo de Boosting

A cada iteração $t$, adicionamos a árvore que melhor corrija os **pseudo-resíduos** da iteração anterior:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(\mathbf{x}_i)$$

onde $\eta$ (learning rate) controla o passo. Valores menores exigem mais árvores mas geralmente generalizam melhor.

### XGBoost: Contribuições Técnicas

O **XGBoost** (Chen & Guestrin, 2016) introduz várias melhorias sobre o gradient boosting clássico:

1. **Regularização explícita na função objetivo:**

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} l(y_i, \hat{y}_i^{(t)}) + \Omega(f_t)$$

onde $\Omega(f) = \gamma T + \frac{1}{2}\lambda \|\mathbf{w}\|^2$ penaliza número de folhas $T$ e magnitude dos pesos $\mathbf{w}$.

2. **Aproximação de segunda ordem (Newton boosting):** usa tanto o gradiente $g_i$ quanto a Hessiana $h_i$ para calcular os splits ótimos analiticamente.

3. **Column subsampling** (`colsample_bytree`): amostragem aleatória de features por árvore — reduz correlação entre árvores, semelhante ao Random Forest.

4. **Implementação eficiente:** suporte nativo a matrizes esparsas, algoritmo de split aproximado para grandes datasets.

### Hiperparâmetros Adotados

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `n_estimators` | `500` | Árvores suficientes para convergência |
| `max_depth` | `6` | Moderado — evita overfitting em features esparsas |
| `learning_rate` | `0.05` | Conservador — melhor generalização |
| `subsample` | `0.8` | Amostragem de 80% das linhas por árvore |
| `colsample_bytree` | `0.8` | Amostragem de 80% das features por árvore |
| `min_child_weight` | `3` | Evita splits em nós com poucos exemplos |
| `eval_metric` | `'logloss'` | Métrica de treino para classificação binária |

## 6.1 Funções Auxiliares de Avaliação

Reutilizamos o mesmo padrão de avaliação do Nível 1.

In [ ]:
def evaluate_model(model, X, y_true, name):
    y_pred = model.predict(X)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X)[:, 1]
    else:
        y_score = model.decision_function(X)

    metrics = {
        'model':       name,
        'accuracy':    accuracy_score(y_true, y_pred),
        'f1_macro':    f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'precision':   precision_score(y_true, y_pred, average='macro'),
        'recall':      recall_score(y_true, y_pred, average='macro'),
        'roc_auc':     roc_auc_score(y_true, y_score),
        'y_pred':      y_pred,
        'y_score':     y_score,
    }

    print(f'{"=" * 56}')
    print(f'  {name}')
    print(f'{"=" * 56}')
    print(f'  Accuracy         : {metrics["accuracy"]:.4f}')
    print(f'  F1 Macro         : {metrics["f1_macro"]:.4f}')
    print(f'  F1 Weighted      : {metrics["f1_weighted"]:.4f}')
    print(f'  Precision Macro  : {metrics["precision"]:.4f}')
    print(f'  Recall Macro     : {metrics["recall"]:.4f}')
    print(f'  ROC-AUC          : {metrics["roc_auc"]:.4f}')
    print()
    print(classification_report(y_true, y_pred,
                                 target_names=['Fake (0)', 'Real (1)'],
                                 digits=4))
    return metrics


def plot_cm(y_true, y_pred, title, ax):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    labels  = np.array([[f'{cm[i,j]:,}\n{cm_norm[i,j]:.1%}'
                          for j in range(2)] for i in range(2)])
    sns.heatmap(cm_norm, annot=labels, fmt='', cmap='Blues',
                xticklabels=['Fake', 'Real'],
                yticklabels=['Fake', 'Real'],
                ax=ax, vmin=0, vmax=1, annot_kws={'size': 11})
    ax.set_title(title)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')


print('Funções auxiliares definidas.')

## 7. Modelo A — XGBoost Estilométrico (22 features)

Treinamos o XGBoost **apenas com as 22 features estilométricas**, sem nenhuma informação de vocabulário. Isso responde à pergunta: *"o estilo de escrita, por si só, consegue distinguir fake news?"*

In [ ]:
# ── Validação Cruzada 5-Fold — Modelo A ───────────────────────────────────────
print('Executando validação cruzada 5-fold — XGBoost Estilométrico...')

xgb_styl = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    verbosity=0,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_styl = cross_validate(
    xgb_styl, X_train_styl, y_train,
    cv=cv,
    scoring=['accuracy', 'f1_macro', 'roc_auc'],
    return_train_score=True,
    n_jobs=-1
)

print('\n=== Cross-Validation (5-Fold) — XGBoost Estilométrico ===')
print(f'  {"Métrica":<16} | {"Val (média ± std)":>22} | {"Train (média)":>15}')
print('  ' + '-' * 60)
for m in ['accuracy', 'f1_macro', 'roc_auc']:
    v = cv_styl[f'test_{m}']
    t = cv_styl[f'train_{m}']
    print(f'  {m:<16} | {v.mean():.4f} ± {v.std():.4f}          | {t.mean():.4f}')

In [ ]:
# ── Treinamento final ─────────────────────────────────────────────────────────
t0 = time.time()
xgb_styl.fit(X_train_styl, y_train)
styl_time = time.time() - t0
print(f'XGBoost Estilométrico treinado em {styl_time:.2f}s')

# ── Avaliação no Teste ────────────────────────────────────────────────────────
styl_res = evaluate_model(xgb_styl, X_test_styl, y_test, 'XGBoost Estilométrico')
styl_res['train_time'] = styl_time

In [ ]:
# ── Visualizações — Modelo A ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_cm(y_test, styl_res['y_pred'], 'Confusion Matrix — XGBoost Estilométrico', axes[0])

fpr_s, tpr_s, _ = roc_curve(y_test, styl_res['y_score'])
axes[1].plot(fpr_s, tpr_s, color=PALETTE['neutral'], lw=2,
             label=f'XGBoost Estilométrico  (AUC={styl_res["roc_auc"]:.4f})')
axes[1].fill_between(fpr_s, tpr_s, alpha=0.07, color=PALETTE['neutral'])
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatório')
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('Curva ROC — XGBoost Estilométrico')
axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgb_stylometric_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Modelo B — XGBoost Híbrido (TF-IDF + Estilométrico)

Combinamos a representação TF-IDF (100.000 features de vocabulário) com as 22 features estilométricas em uma única matriz de `100.022` features. O XGBoost aprende automaticamente quais features — de vocabulário ou de estilo — são mais discriminativas para cada split.

**Hipótese:** as features estilométricas capturam variância que o TF-IDF deixa passar, pois TF-IDF ignora por completo pontuação, capitalização e estrutura de sentenças.

In [ ]:
# ── Validação Cruzada 5-Fold — Modelo B ───────────────────────────────────────
print('Executando validação cruzada 5-fold — XGBoost Híbrido...')

xgb_hybrid = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    verbosity=0,
)

cv_hybrid = cross_validate(
    xgb_hybrid, X_train_hybrid, y_train,
    cv=cv,
    scoring=['accuracy', 'f1_macro', 'roc_auc'],
    return_train_score=True,
    n_jobs=-1
)

print('\n=== Cross-Validation (5-Fold) — XGBoost Híbrido ===')
print(f'  {"Métrica":<16} | {"Val (média ± std)":>22} | {"Train (média)":>15}')
print('  ' + '-' * 60)
for m in ['accuracy', 'f1_macro', 'roc_auc']:
    v = cv_hybrid[f'test_{m}']
    t = cv_hybrid[f'train_{m}']
    print(f'  {m:<16} | {v.mean():.4f} ± {v.std():.4f}          | {t.mean():.4f}')

In [ ]:
# ── Treinamento final ─────────────────────────────────────────────────────────
t0 = time.time()
xgb_hybrid.fit(X_train_hybrid, y_train)
hybrid_time = time.time() - t0
print(f'XGBoost Híbrido treinado em {hybrid_time:.2f}s')

# ── Avaliação no Teste ────────────────────────────────────────────────────────
hybrid_res = evaluate_model(xgb_hybrid, X_test_hybrid, y_test, 'XGBoost Híbrido')
hybrid_res['train_time'] = hybrid_time

In [ ]:
# ── Visualizações — Modelo B ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_cm(y_test, hybrid_res['y_pred'], 'Confusion Matrix — XGBoost Híbrido', axes[0])

fpr_h, tpr_h, _ = roc_curve(y_test, hybrid_res['y_score'])
axes[1].plot(fpr_h, tpr_h, color=PALETTE['real'], lw=2,
             label=f'XGBoost Híbrido  (AUC={hybrid_res["roc_auc"]:.4f})')
axes[1].fill_between(fpr_h, tpr_h, alpha=0.07, color=PALETTE['real'])
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatório')
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('Curva ROC — XGBoost Híbrido')
axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgb_hybrid_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Análise SHAP — Explicabilidade do Modelo Estilométrico

### O que é SHAP?

**SHAP (SHapley Additive exPlanations)** é um framework unificado de explicabilidade baseado na teoria dos jogos cooperativos (Lundberg & Lee, 2017). Para cada predição, o SHAP calcula a contribuição de cada feature — os **Shapley values**.

#### Shapley Values

Para um jogador $i$ em um jogo cooperativo, o Shapley value é:

$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[v(S \cup \{i\}) - v(S)\right]$$

onde $F$ é o conjunto de todas as features, $S$ é uma coalização de features sem $i$, e $v(S)$ é a predição do modelo usando apenas as features em $S$.

Traduzindo: $\phi_i$ mede a **contribuição marginal média** de incluir a feature $i$ em todas as possíveis combinações de features.

#### Propriedades Fundamentais

| Propriedade | Significado |
|-------------|-------------|
| **Eficiência** | $\sum_i \phi_i = f(\mathbf{x}) - \mathbb{E}[f]$ — valores somam o desvio da predição em relação à média |
| **Simetria** | Features com igual contribuição recebem igual SHAP value |
| **Dummy** | Feature que não afeta predição tem SHAP = 0 |
| **Aditividade** | Ensemble de modelos → soma dos SHAP values |

#### TreeExplainer

Para modelos de árvore (incluindo XGBoost), o **TreeExplainer** calcula os Shapley values **exatos** em $O(T \cdot L^2)$, onde $T$ é o número de árvores e $L$ é o número de folhas. É muito mais eficiente que a aproximação por amostragem usada para outros modelos.

### Interpretação do Beeswarm Plot

- **Eixo X**: valor SHAP — negativo empurra para Fake, positivo empurra para Real
- **Eixo Y**: features ordenadas por importância global ($|\phi|$ médio)
- **Cor do ponto**: valor real da feature — vermelho=alto, azul=baixo
- **Cada ponto**: um artigo do conjunto de teste

In [ ]:
# ── TreeExplainer sobre o Modelo A (estilométrico) ─────────────────────────────
print('Calculando SHAP values (TreeExplainer)...')
t0 = time.time()

explainer  = shap.TreeExplainer(xgb_styl)
shap_vals  = explainer.shap_values(X_test_styl)

# Em alguns casos o TreeExplainer retorna lista [classe_0, classe_1]
# Para XGBoost binário, pegamos a classe positiva (Real)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]

print(f'Concluído em {time.time()-t0:.1f}s')
print(f'Shape dos SHAP values: {shap_vals.shape}')

# DataFrame do conjunto de teste com nomes de features
X_test_styl_df = pd.DataFrame(X_test_styl, columns=STYL_FEATURES)

In [ ]:
# ── Importância global (|SHAP| médio) ─────────────────────────────────────────
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=STYL_FEATURES).sort_values(ascending=False)

print('=== Importância Global das Features (|SHAP| médio) ===')
for feat, val in shap_importance.items():
    bar = '█' * int(val / shap_importance.max() * 30)
    print(f'  {feat:<30s}  {val:.4f}  {bar}')

In [ ]:
# ── Beeswarm Plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals,
    X_test_styl_df,
    show=False,
    plot_size=None,
    max_display=22
)
plt.title('SHAP Beeswarm — XGBoost Estilométrico\n'
          '(negativo → Fake, positivo → Real)',
          fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Bar Plot de Importância ────────────────────────────────────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_vals,
    X_test_styl_df,
    plot_type='bar',
    show=False,
    plot_size=None,
    max_display=22
)
plt.title('SHAP Feature Importance — XGBoost Estilométrico\n'
          '(|SHAP| médio = contribuição média ao score de predição)',
          fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Comparação com Nível 1 — Benchmark Completo

Comparamos todos os modelos treinados ao longo da pipeline, usando o **mesmo conjunto de teste** para garantir comparação justa.

In [ ]:
# ── Resultados do Nível 1 (valores do conjunto de teste reportados no NB 01) ──
# Estes valores foram obtidos com o mesmo split (RANDOM_STATE=42, 70/15/15)
LVL1_BENCHMARK = [
    {'model': 'LR  (Nível 1)',  'f1_macro': 0.9878, 'roc_auc': 0.9991,
     'accuracy': 0.9878, 'train_time': 0.81},
    {'model': 'SVM (Nível 1)',  'f1_macro': 0.9939, 'roc_auc': 0.9996,
     'accuracy': 0.9939, 'train_time': 4.74},
]

# ── Tabela consolidada ────────────────────────────────────────────────────────
rows = LVL1_BENCHMARK.copy()
for res in [styl_res, hybrid_res]:
    rows.append({
        'model':      res['model'],
        'accuracy':   res['accuracy'],
        'f1_macro':   res['f1_macro'],
        'roc_auc':    res['roc_auc'],
        'train_time': res['train_time'],
    })

comp_df = pd.DataFrame(rows)
comp_df['Nível'] = ['1', '1', '2-A', '2-B']
comp_df = comp_df.set_index('model')

print('=== Comparação Completa — Conjunto de Teste ===')
print(comp_df[['Nível', 'accuracy', 'f1_macro', 'roc_auc', 'train_time']]
      .to_string(float_format='{:.4f}'.format))

# Destaque do ganho sobre o benchmark do Nível 1
best_lvl1 = 0.9939
best_lvl2 = hybrid_res['f1_macro']
delta = best_lvl2 - best_lvl1
print(f'\n  Melhor F1 Macro Nível 1 (LinearSVC) : {best_lvl1:.4f}')
print(f'  Melhor F1 Macro Nível 2 (XGB Híbr.) : {best_lvl2:.4f}')
print(f'  Delta                                : {delta:+.4f} ({delta*100:+.2f} pp)')

if best_lvl2 > best_lvl1:
    print('  ✓ Nível 2 supera o benchmark do Nível 1.')
else:
    print('  ✗ Nível 2 não supera o benchmark do Nível 1 — revisar hiperparâmetros.')

In [ ]:
# ── Gráfico de comparação ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names  = ['LR\n(N1)', 'SVM\n(N1)', 'XGB-Styl\n(N2-A)', 'XGB-Hyb\n(N2-B)']
f1_vals      = [r['f1_macro']  for r in rows]
auc_vals     = [r['roc_auc']   for r in rows]
bar_colors   = [PALETTE['neutral'], PALETTE['neutral'],
                PALETTE['fake'],    PALETTE['real']]

# F1 Macro
bars = axes[0].bar(model_names, f1_vals, color=bar_colors,
                   alpha=0.85, edgecolor='black', lw=0.7)
for bar, val in zip(bars, f1_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].axhline(best_lvl1, color='gray', linestyle='--', lw=1.2,
                label=f'Benchmark N1 = {best_lvl1:.4f}')
axes[0].set_ylim([0.87, 1.005])
axes[0].set_ylabel('F1 Macro')
axes[0].set_title('F1 Macro — todos os modelos')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# ROC-AUC
bars2 = axes[1].bar(model_names, auc_vals, color=bar_colors,
                    alpha=0.85, edgecolor='black', lw=0.7)
for bar, val in zip(bars2, auc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.0002,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[1].set_ylim([0.97, 1.002])
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('ROC-AUC — todos os modelos')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Comparação de Modelos: Nível 1 vs Nível 2', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Conclusões

### Resultados

| Modelo | Features | F1 Macro | ROC-AUC |
|--------|----------|----------|---------|
| LR Nível 1 | TF-IDF (100k) | 0.9878 | 0.9991 |
| LinearSVC Nível 1 | TF-IDF (100k) | 0.9939 | 0.9996 |
| **XGBoost Estilométrico** | 22 features | *ver saída* | *ver saída* |
| **XGBoost Híbrido** | TF-IDF + 22 | *ver saída* | *ver saída* |

---

### Insights do SHAP

A análise SHAP revelou as features estilométricas mais discriminativas. As features esperadas com maior importância:

**Associadas a REAL:**
- `unique_word_ratio` alto → vocabulário diverso e rico (Reuters)
- `avg_word_len` alto → linguagem técnica e formal
- `avg_sent_len` alto → sentenças longas e complexas
- `number_ratio` alto → dados quantitativos, estatísticas

**Associadas a FAKE:**
- `exclamation_count` alto → sensacionalismo
- `caps_word_ratio` alto → ALL CAPS para ênfase emocional
- `url_count` alto → referências a fontes externas não-jornalísticas
- `title_has_exclamation` = 1 → títulos clickbait

---

### Limitações do Nível 2

1. **Features estilométricas são fracas em isolamento**: o estilo por si só captura padrões gerais mas perde poder quando o texto é mais neutro
2. **XGBoost + TF-IDF esparso**: 100k features esparsas em gradient boosting pode ser subótimo — modelos lineares (LinearSVC) tendem a explorar TF-IDF melhor
3. **Sem semântica**: nenhum modelo de Nível 1 ou 2 entende significado — "bank" (margem de rio) e "bank" (banco financeiro) são o mesmo token

---

### Próximos Passos

```
Nível 3 → BiLSTM + GloVe / TextCNN
           Dependências sequenciais, embeddings pré-treinados
           Meta: superar F1 Macro do melhor modelo de Nível 2

Nível 4 → DistilBERT / RoBERTa Fine-tuning
           Representações contextuais profundas — estado da arte

Nível 5 → Ensemble Heterogêneo
           Stacking dos melhores modelos de cada nível
```

---

### Referências

- Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD.
- Lundberg, S.M. & Lee, S. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS.
- Shapley, L.S. (1953). *A value for n-person games*. Contributions to the Theory of Games.
- Rashkin, H. et al. (2017). *Truth of Varying Shades: Analyzing Language in Fake News*. EMNLP.
- Pérez-Rosas, V. et al. (2018). *Automatic Detection of Fake News*. COLING.